## Fonemización del dataset

In [4]:
import phonemizer

global_phonemizer_valencia = phonemizer.backend.EspeakBackend(
    language="ca-va",
    preserve_punctuation=True,
    with_stress=True,
    language_switch="remove-flags",
)

# Esto no funciona si no configurar previamente el backend: espeak.
# Para configurarlo hay que usar el repo espeak-ng (el base ya tiene el valenciano) o el de aina, son equivalentes.
# Puedes echar un vistazo a: https://huggingface.co/spaces/projecte-aina/transcripcio-fonetica-catala/blob/main/Dockerfile


def phonemize(texts):
  output = global_phonemizer_valencia.phonemize(texts, strip=True, njobs=1)

  return output

print(phonemize(["Hola mundo", "Com va tot?"]))

['ˈɔla mˈundo', 'kˈoɱ vˈa tˈot?']


In [5]:
import os
os.environ['HF_HOME'] = '/media/datos1/tts-valenciano/huggingface'
os.environ['HF_DATASETS_CACHE'] = '/media/datos1/tts-valenciano/huggingface/datasets'

print(f"HF_HOME establecido en: {os.environ['HF_HOME']}")
print(f"HF_DATASETS_CACHE establecido en: {os.environ['HF_DATASETS_CACHE']}")

# Ahora sí importar las librerías
from datasets import load_dataset
import pandas as pd
import numpy as np
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()

login(os.getenv("HUGGINGFACE_TOKEN"))

# TODO: Esto podríamos moverlo a un script aparte ya que lo hacemos igual en varios sitios
print("\nCargando dataset...")
dataset = load_dataset("javiimts/corts-valencianes-asr-parquet")
print(f"Splits disponibles: {dataset.keys()}")

HF_HOME establecido en: /media/datos1/tts-valenciano/huggingface
HF_DATASETS_CACHE establecido en: /media/datos1/tts-valenciano/huggingface/datasets

Cargando dataset...


Resolving data files:   0%|          | 0/81 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/77 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/81 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/77 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/71 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/68 [00:00<?, ?it/s]

Splits disponibles: dict_keys(['clean_train_short', 'clean_test_short', 'clean_dev_short', 'other_train_short', 'other_test_short', 'other_dev_short', 'clean_train_long', 'clean_test_long', 'clean_dev_long', 'other_train_long', 'other_test_long', 'other_dev_long'])


In [6]:
# Procesar todos los splits del dataset
# Primero vemos qué splits hay disponibles
print("Splits disponibles:")
for split_name in dataset.keys():
    print(f"  - {split_name}: {len(dataset[split_name])} muestras")

Splits disponibles:
  - clean_train_short: 44932 muestras
  - clean_test_short: 1325 muestras
  - clean_dev_short: 1284 muestras
  - other_train_short: 44399 muestras
  - other_test_short: 1467 muestras
  - other_dev_short: 1267 muestras
  - clean_train_long: 1286 muestras
  - clean_test_long: 31 muestras
  - clean_dev_long: 31 muestras
  - other_train_long: 1390 muestras
  - other_test_long: 21 muestras
  - other_dev_long: 27 muestras


In [7]:
# Explorar la estructura del dataset
split = "clean_dev_short"
print(f"\nInformación del split '{split}':")

print(dataset[split])

# Ver las columnas
print("\nColumnas del dataset:")
print(dataset[split].column_names)
# Ver un ejemplo
print("\nPrimer ejemplo:")
print(dataset[split][0])


Información del split 'clean_dev_short':
Dataset({
    features: ['identifier', 'audio', 'segment_path', 'text'],
    num_rows: 1284
})

Columnas del dataset:
['identifier', 'audio', 'segment_path', 'text']

Primer ejemplo:
{'identifier': '7767db7ab74c86e79bb141dbb9f31235', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x765362f124a0>, 'segment_path': 'corpus/speech/clean_dev_short/7/7/6/7767db7ab74c86e79bb141dbb9f31235.wav', 'text': "d'una banda la generalitat que pot costar nos més de mil milions d'euros un aval que vam assumir totes les valencianes i valencians per decisió del pp quan els beneficis de la fira eren clarament insuficients i el govern del pp ho sabia però ho ocultava i obstaculitzava la seua fiscalització negant una comissió d'investigació que avui s'aprovarà"}


## Conversión del dataset al formato StyleTTS2

El dataset tiene las siguientes columnas:
- `identifier`: ID único del audio
- `audio`: objeto de audio
- `segment_path`: path original del segmento
- `text`: transcripción del audio

Necesitamos crear:
1. Una carpeta `wavs/` con todos los archivos de audio
2. Archivos txt con formato: `wavs/audio_name.wav|fonemas|speaker_id`

In [ ]:
# Configuración de directorios
import os
from pathlib import Path

# Directorio base donde se guardará todo
output_dir = Path("/media/datos1/tts-valenciano/styletts2_corts_dataset")
wavs_dir = output_dir / "wavs"

# Crear directorios si no existen
wavs_dir.mkdir(parents=True, exist_ok=True)

print(f"Directorio de salida: {output_dir}")
print(f"Directorio de audios: {wavs_dir}")

Directorio de salida: /media/datos1/tts-valenciano/styletts2_corts_dataset
Directorio de audios: /media/datos1/tts-valenciano/styletts2_corts_dataset/wavs


In [ ]:
import soundfile as sf
import numpy as np
from datasets import Dataset
import os

def _extract_audio_and_sr(audio_field):
    return audio_field['array'], audio_field['sampling_rate']


def map_generate_batched(batch):
    identifiers = batch['identifier']
    texts = batch['text']
    audios = batch['audio']
    
    # 1. Fonemizar el batch de textos de una sola vez
    try:
        batch_phonemes = phonemize(texts)
    except Exception as e:
        print(f"Error en phonemize (batch): {e}. Textos: {texts}")
        batch_phonemes = [""] * len(texts) # Fallback
    
    lines = []
    # 2. Iterar sobre el batch PRE-PROCESADO
    for ident, aud, phonemes in zip(identifiers, audios, batch_phonemes):
        
        # 3. Extraer audio y SR
        audio_array, sr = _extract_audio_and_sr(aud)
        
        # 4. Validar y guardar
        if audio_array is None or sr is None or not phonemes:
            lines.append("") # Se descarta
            continue
            
        wav_name = f"{ident}.wav"
        sf.write(str(wavs_dir / wav_name), audio_array, int(sr))
        
        speaker_id = "0"
        line = f"wavs/{wav_name}|{phonemes}|{speaker_id}"
        lines.append(line)
        
    return {"styletts2_line": lines}

print("Funciones para procesamiento con map (versión batched) listas.")

Funciones para procesamiento con map (versión batched) listas.


## Mapeo de splits a archivos StyleTTS2

StyleTTS2 necesita 3 archivos:
- **train_list.txt**: Datos de entrenamiento (el split más grande)
- **val_list.txt**: Datos de validación (para evaluar durante el entrenamiento)
- **OOD_texts.txt**: Textos "Out Of Distribution" (textos para probar la generalización del modelo)

In [51]:
# Mapeo de splits a archivos StyleTTS2
# Basándonos en los splits disponibles:

# train_list.txt: Usaremos los splits de train (clean y other)
train_splits = ['clean_train_short', 'other_train_short']

# val_list.txt: Usaremos los splits de test (clean y other)
val_splits = ['clean_test_short', 'other_test_short']

# OOD_texts.txt: Usaremos los splits de dev (clean y other) 
# Estos serán textos "Out Of Distribution" para probar generalización
ood_splits = ['clean_dev_short', 'other_dev_short']

# Resumen del mapeo
print("Mapeo de splits a archivos StyleTTS2:\n")
print(f"train_list.txt ({sum(len(dataset[s]) for s in train_splits)} muestras):")
for split in train_splits:
    print(f"  - {split}: {len(dataset[split])} muestras")

print(f"\nval_list.txt ({sum(len(dataset[s]) for s in val_splits)} muestras):")
for split in val_splits:
    print(f"  - {split}: {len(dataset[split])} muestras")

print(f"\nOOD_texts.txt ({sum(len(dataset[s]) for s in ood_splits)} muestras):")
for split in ood_splits:
    print(f"  - {split}: {len(dataset[split])} muestras")

print(f"\nTotal de muestras: {sum(len(dataset[s]) for s in train_splits + val_splits + ood_splits)}")

Mapeo de splits a archivos StyleTTS2:

train_list.txt (89331 muestras):
  - clean_train_short: 44932 muestras
  - other_train_short: 44399 muestras

val_list.txt (2792 muestras):
  - clean_test_short: 1325 muestras
  - other_test_short: 1467 muestras

OOD_texts.txt (2551 muestras):
  - clean_dev_short: 1284 muestras
  - other_dev_short: 1267 muestras

Total de muestras: 94674


In [ ]:
import os
from datasets import concatenate_datasets

NUM_PROCS = max(1, os.cpu_count() - 1) 
# Tamaño del batch para cada proceso. 32 o 64 suele ir bien aquí en este servidor
BATCH_SIZE = 64

print(f"Usando {NUM_PROCS} procesos para .map()")

def process_and_save_splits(split_names, output_filename):
    """
    Toma una lista de nombres de split, los procesa en paralelo con map,
    combina los resultados y guarda el archivo .txt final.
    """
    print(f"\n" + "="*60)
    print(f"PROCESANDO: {output_filename}")
    print(f"Splits a procesar: {split_names}")
    print("="*60)
    
    # 1. Cargar y concatenar los splits si hay más de uno
    dsets_to_process = [dataset[s] for s in split_names]
    full_ds = concatenate_datasets(dsets_to_process)
    print(f"Total de muestras para '{output_filename}': {len(full_ds)}")
    
    # 2. Aplicar el map en paralelo
    print(f"Ejecutando .map() con {NUM_PROCS} procesos...")
    processed_ds = full_ds.map(
        map_generate_batched,
        batched=True,
        batch_size=BATCH_SIZE,
        num_proc=NUM_PROCS
    )
    
    # 3. Recolectar las líneas y filtrar vacías
    all_lines = [line for line in processed_ds["styletts2_line"] if line]
    
    # 4. Guardar el archivo .txt
    output_file = output_dir / output_filename
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(all_lines))
        
    print(f"\n✓ {output_filename} guardado: {len(all_lines)} muestras totales")
    return all_lines

# 1. Procesar train_list.txt
train_lines = process_and_save_splits(train_splits, "train_list.txt")

# 2. Procesar val_list.txt
val_lines = process_and_save_splits(val_splits, "val_list.txt")

# 3. Procesar OOD_texts.txt
ood_lines = process_and_save_splits(ood_splits, "OOD_texts.txt")


# Resumen final
print("\n" + "="*60)
print("RESUMEN FINAL")
print("="*60)
print(f"Directorio: {output_dir}")
print(f"  - train_list.txt: {len(train_lines)} muestras")
print(f"  - val_list.txt: {len(val_lines)} muestras")
print(f"  - OOD_texts.txt: {len(ood_lines)} muestras")
print(f"  - Total audios en wavs/: {len(train_lines) + len(val_lines) + len(ood_lines)}")
print("\n✓ Conversión del dataset completada (con .map())!")

Usando 7 procesos para .map()

PROCESANDO: train_list.txt
Splits a procesar: ['clean_train_short', 'other_train_short']
Total de muestras para 'train_list.txt': 89331
Ejecutando .map() con 7 procesos...
Total de muestras para 'train_list.txt': 89331
Ejecutando .map() con 7 procesos...


Map (num_proc=7):   0%|          | 0/89331 [00:00<?, ? examples/s]


✓ train_list.txt guardado: 89331 muestras totales

PROCESANDO: val_list.txt
Splits a procesar: ['clean_test_short', 'other_test_short']
Total de muestras para 'val_list.txt': 2792
Ejecutando .map() con 7 procesos...


Map (num_proc=7):   0%|          | 0/2792 [00:00<?, ? examples/s]


✓ val_list.txt guardado: 2792 muestras totales

PROCESANDO: OOD_texts.txt
Splits a procesar: ['clean_dev_short', 'other_dev_short']
Total de muestras para 'OOD_texts.txt': 2551
Ejecutando .map() con 7 procesos...


Map (num_proc=7):   0%|          | 0/2551 [00:00<?, ? examples/s]


✓ OOD_texts.txt guardado: 2551 muestras totales

RESUMEN FINAL
Directorio: /media/datos1/tts-valenciano/styletts2_corts_dataset
  - train_list.txt: 89331 muestras
  - val_list.txt: 2792 muestras
  - OOD_texts.txt: 2551 muestras
  - Total audios en wavs/: 94674

✓ Conversión del dataset completada (con .map())!
